# Streaming chunked public audio

Executed on CPU with the real checkpoint. Uses public FLEURS validation audio at a pinned revision. Audio and transcripts are not embedded in outputs. No performance or accuracy estimate is derived from these two rows. Decoder context above eight seconds is **unvalidated**.

Install from a checkout with `pip install '.[torch,datasets]'` and compatible TorchCodec/FFmpeg, plus `nbclient nbformat ipykernel`. Authenticate outside this notebook if required. Converted head/readout weights are Apache-2.0.


In [1]:
import torch
from datasets import Audio, load_dataset

from marimba import Marimba, Question

torch.set_num_threads(4)
revision = "70bb2e84b976b7e960aa89f1c648e09c59f894dd"
url = (
    "hf://datasets/google/fleurs@"
    + revision
    + "/parquet-data/en_us/validation-00000-of-00001.parquet"
)
ds = load_dataset("parquet", data_files={"validation": url}, split="validation[:2]")
ds = ds.cast_column("audio", Audio(sampling_rate=16000))
model = Marimba.from_pretrained(
    revision="bc980b4e6eb34e7fea96ea8b0884c2093a845dc5",
    backend="torch",
    device="cpu",
    dtype="float32",
)
print(
    {
        "dataset": "google/fleurs",
        "revision": revision,
        "split": "validation",
        "rows": len(ds),
        "backend": model.runtime.backend,
        "device": model.runtime.device,
        "dtype": model.runtime.dtype,
    }
)

{'dataset': 'google/fleurs', 'revision': '70bb2e84b976b7e960aa89f1c648e09c59f894dd', 'split': 'validation', 'rows': 2, 'backend': 'torch', 'device': 'cpu', 'dtype': 'float32'}


In [2]:
import numpy as np

from marimba.audio import decode_audio

samples = decode_audio(ds[0]["audio"])
questions = [Question.noul("Is someone speaking?")]
stream = model.stream(questions, hop_seconds=1.0, coalesce=True)
updates = []
for start in range(0, len(samples), 4000):
    updates.extend(stream.push(samples[start : start + 4000]))
updates.extend(stream.flush())
print(
    {
        "sample_rate": 16000,
        "chunk_samples": 4000,
        "observed_seconds": round(len(samples) / 16000, 3),
        "updates": len(updates),
        "last_end_s": round(updates[-1].window_end_s, 3),
        "last_probability": round(updates[-1].decisions[0].probability, 6),
    }
)
assert updates and np.isclose(updates[-1].window_end_s, len(samples) / 16000)

{'sample_rate': 16000, 'chunk_samples': 4000, 'observed_seconds': 6.54, 'updates': 7, 'last_end_s': 6.54, 'last_probability': 0.843403}
